# keras_tuner
- Keras Tuner is a libary that automates the search. Instead of manually eiditng the learning rate and re-running, we define a range of possible values and let the tuner explore them, training a model for each combination and keeping track of which performs best on the validation metrics.

In [ ]:
# import packages
import numpy as np


import pandas as pd
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

In [2]:
%%capture 
# don't show download process
# download the mnist data from datasets
(img_train, label_train),(img_test, label_test) = keras.datasets.fashion_mnist.load_data()

In [3]:
# scaling data between 0 and 1
img_train = img_train.astype('float32') / 255.0
img_test = img_test.astype('float32') / 255.0

In [ ]:
# build a model function
def model_builder(hp):
    model = keras.Sequential()
    model.add(keras.layers.Flatten(input_shape=(28,28))) # same as keras.layers.Input(shape=(28,28)); Dense layer expects 1D, so need to use .Flatten()

    # tune the number of units in the first Dense layer
    # choose an optimal value between 32-512
    hy_units = hp.Int('units', min_value=32, max_value=512, step=32)
    model.add(keras.layers.Dense(units=hy_units, activation='relu'))
    model.add(keras.layers.Dense(10)) # no activation function

    # Tune the learning rate for the optimizer
    # Choose an optimal value from 0.01, 0.001, 0r 0.0001
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)

    # before layer outputs raw logits, the loss funciton applies softmax internally before computing cross-entropy
    # Cross-entropy loss fundamentally needs probabilities to do its computation
    loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True) 

    model.compile(optimizer = optimizer,
                  loss = loss,
                  metrics=['accuracy'])

    return model

In [ ]:
%%capture
# initialize the tuner and perform hypertuning using Hyperbank algorithm - random search.
# it trains many configurations for a few epochs, kills the week ones early, and gives the survivors more budget. It 
# makes much faster than grid search for finding good hyperparameters
tuner = kt.Hyperband(
    model_builder, # the model with hp auguments, return a compiled Keras model
    objective='val_accuracy', # evaluation metrics
    max_epochs=10, # maximum epochs the final hyperparameters run based all survived configurations
    factor=3, # reduction factor.  at each round of a bracket, Hyperband keeps roughly the to 1/factor of configurations
    directory='my_dir', # the folder on disk where the tuner saves its state
    project_name='intro_to_k', # a subfolder name insider directory that namespaces this particular search
    seed=0, 
    overwrite=True # regenerate
)


In [ ]:
# creates a callback that halts a model's training when it stops improving
# monitor - the metrics it watchs
# patience = it tolerates 5 consecutive epochs whith no improvement in val_loss before stopping that run
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

In [ ]:
# search over all configurations
tuner.search(
    img_train, # features
    label_train, # labels
    epochs=5, # 
    validation_split=0.2,
    callbacks=[early_stop])

Trial 30 Complete [00h 00m 16s]
val_accuracy: 0.877916693687439

Best val_accuracy So Far: 0.8916666507720947
Total elapsed time: 00h 03m 16s


In [20]:
# return the top-ranked 1 hyperparameter configurations, return a list
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

In [ ]:
# best 'units'
best_hps.get('units')

352

In [ ]:
# best 'learning_rate' 
best_hps.get('learning_rate')

0.001

In [ ]:
# use the best hyperparameters to build the model
model = tuner.hypermodel.build(best_hps)

In [ ]:
# fit the training data
history = model.fit(img_train, label_train, epochs=5, batch_size= 300, validation_split=0.2)

Epoch 1/5
160/160 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7969 - loss: 0.5912 - val_accuracy: 0.8268 - val_loss: 0.4880
Epoch 2/5
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8561 - loss: 0.4140 - val_accuracy: 0.8603 - val_loss: 0.3975
Epoch 3/5
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8689 - loss: 0.3759 - val_accuracy: 0.8716 - val_loss: 0.3751
Epoch 4/5
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8763 - loss: 0.3477 - val_accuracy: 0.8734 - val_loss: 0.3616
Epoch 5/5
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8824 - loss: 0.3286 - val_accuracy: 0.8702 - val_loss: 0.3716


In [ ]:
# get the best epochs
val_acc_per_epch = history.history['val_accuracy']
best_epoch = val_acc_per_epch.index(max(val_acc_per_epch)) + 1

In [ ]:
# build the model using the best hyperparameter and the best epochs
hypermodel = tuner.hypermodel.build(best_hps)

In [ ]:
# refit the model
hypermodel.fit(img_train, label_train, epochs=best_epoch, batch_size=300, validation_split=0.2)

Epoch 1/4
160/160 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7966 - loss: 0.5975 - val_accuracy: 0.8450 - val_loss: 0.4573
Epoch 2/4
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8544 - loss: 0.4181 - val_accuracy: 0.8598 - val_loss: 0.4042
Epoch 3/4
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8664 - loss: 0.3780 - val_accuracy: 0.8651 - val_loss: 0.3776
Epoch 4/4
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8748 - loss: 0.3493 - val_accuracy: 0.8701 - val_loss: 0.3657


In [ ]:
# return the model performance
eval_result = hypermodel.evaluate(img_test, label_test)
eval_result

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 324us/step - accuracy: 0.8605 - loss: 0.3871


[0.3870704472064972, 0.8604999780654907]